In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import pickle

In [13]:
import os

# Check if model file exists before loading
model_path = './ANN/churn_model.h5'
if os.path.exists(model_path):
    model = load_model(model_path)
else:
    print(f"Model file '{model_path}' not found. Please check the path or filename.")

# load encoder and scaler
path = './ANN/label_encodergender.pkl'
with open(path, 'rb') as f:
    label_encoder = pickle.load(f)

with open('./ANN/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('./ANN/one_hot_encoder_geography.pkl', 'rb') as f:
    one_hot_encoder = pickle.load(f)

In [8]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',   
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1, 
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [ ]:
geonecoded = one_hot_encoder.transform([[input_data['Geography']]]).toarray()
geonecoded_def = pd.DataFrame(geonecoded,columns=one_hot_encoder.get_feature_names_out(['Geography']))
geonecoded_def

c:\LearningWS\MachineLearning\mlenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [21]:
input_df = pd.DataFrame([input_data])
input_df

ValueError: Must pass 2-d input. shape=(1, 1, 13)

In [20]:
input_data = pd.concat([input_df.reset_index(drop=True), geonecoded_def],axis = 1)
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [22]:
## encode catagorical 
input_df['Gender'] = label_encoder.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [23]:
## concetination with Goograpgy a
input_df = pd.concat([input_df.drop("Geography", axis = 1),geonecoded_def], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [24]:
## scaling the data
input_scale = scaler.transform(input_df)
input_scale

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [25]:
## predict
prediction = model.predict(input_scale)
prediction

1/1 [==============================] - 0s 155ms/step


array([[0.05680177]], dtype=float32)

In [30]:
pridiction_prob = prediction[0][0]

In [31]:
if( pridiction_prob > 0.5 ):
    print("Customer will likely to exit ")
else:
    print( "Customer will likely not to exit")

Customer will likely not to exit
